# Hierarchical NLI (E-first) — CafeBERT — Google Colab Training Notebook

Clone repo từ GitHub, cài deps, nạp secrets từ Colab Secrets, chọn config (`CONFIG_PATH`), **in toàn bộ config trước khi chạy**, kéo data thật **ViANLI** (uitnlp/ViANLI trên HF Hub), rồi train Hierarchical CafeBERT (Flat baseline mặc định tải lại từ HF Hub, không train lại) với W&B tracking và lưu trữ bắt buộc lên Hugging Face Hub.

**Mỗi cell train/eval đều check exit code và raise lỗi rõ ràng ngay nếu fail** — nếu 1 cell báo lỗi, đừng chạy tiếp cell sau, cuộn lên xem traceback ngay phía trên lỗi đó để biết nguyên nhân thật.

**Trước khi chạy, bắt buộc:**
1. Panel trái → icon 🔑 **Secrets** → thêm 2 secret đúng tên, và **bật "Notebook access"** cho cả 2:
   - `WANDB_API_KEY`
   - `HF_TOKEN`
   (giá trị là 2 key bạn đã cấp trước đó cho W&B và Hugging Face). Notebook sẽ fail rõ ràng ở bước nạp secrets nếu thiếu hoặc chưa bật access.
2. `Runtime` → `Change runtime type` → Hardware accelerator: **GPU** (T4 miễn phí là đủ, A100/L4 nếu có Colab Pro càng nhanh).
3. Kiểm tra `CONFIG_PATH` ở cell bên dưới (mặc định `configs/config_run_a.yaml` = **Run A**: E-first giữ nguyên, chỉ giảm `lambda_fine` 1.0→0.5 + per-example loss — mục tiêu so `dev soft_macro_f1` với kết quả cũ **0.4626**). Đổi thành `configs/config.yaml` nếu muốn chạy lại bản gốc.
4. (Tùy chọn) đổi `DEBUG = True` ở cell bên dưới để chạy thử trên subset 200 mẫu trước khi chạy full — khuyến nghị làm trước để xác nhận toàn bộ pipeline chạy được trước khi đốt thời gian GPU cho full run.

**Lưu ý:** Colab xóa sạch `/content/` mỗi khi ngắt runtime — không sao vì checkpoint/predictions đã push lên **Hugging Face Hub** và metrics đã log lên **W&B** ngay trong lúc chạy, không phụ thuộc vào đĩa cục bộ của Colab.

In [ ]:
# --- GPU check ---
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out or "⚠️ No GPU detected — vào Runtime > Change runtime type, chọn GPU, rồi kết nối lại runtime.")

In [ ]:
# --- Clone toàn bộ code từ GitHub ---
import os, pathlib

REPO_URL = "https://github.com/trantranuit/hierarchical-nli-e-first.git"
REPO_DIR = "/content/hierarchical-nli-e-first"

if pathlib.Path(REPO_DIR).exists():
    !rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
assert _exit_code == 0, "git clone FAILED — kiểm tra kết nối mạng của runtime."
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls -la

In [ ]:
# --- Cài dependencies (torch KHÔNG nằm trong requirements.txt — giữ nguyên
# bản torch Colab đã cài sẵn, đã khớp với GPU của runtime này) ---
!pip install -q -r requirements.txt
assert _exit_code == 0, "pip install FAILED — xem traceback ở trên."

In [ ]:
# --- Sanity check torch/CUDA — test THẬT bằng 1 phép tính trên GPU thay vì đoán qua compute-capability string ---
# (get_arch_list() chỉ liệt kê cubin build sẵn, không hiện PTX forward-compatible — string-match dễ báo sai với GPU đời mới như L4)
import torch
print("torch:", torch.__version__, "| built for CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} (compute capability sm_{props.major}{props.minor})")
    try:
        x = torch.randn(256, 256, device="cuda")
        y = x @ x
        torch.cuda.synchronize()
        print("✓ GPU/torch tương thích (đã chạy thử matmul thật trên GPU), sẵn sàng train.")
    except RuntimeError as e:
        raise RuntimeError(
            f"torch không chạy được kernel CUDA trên {props.name}: {e}\n"
            f"Đổi Accelerator khác (Runtime > Change runtime type), reconnect runtime, rồi chạy lại từ đầu."
        )
else:
    raise RuntimeError("CUDA không available — kiểm tra Runtime > Change runtime type (phải chọn GPU, không phải CPU).")

In [ ]:
# --- Nạp secrets từ Colab Secrets (KHÔNG bao giờ in giá trị thật ra output) ---
from google.colab import userdata
import os

REQUIRED_SECRETS = ["WANDB_API_KEY", "HF_TOKEN"]

loaded, missing = {}, []
for key in REQUIRED_SECRETS:
    try:
        loaded[key] = userdata.get(key)
    except Exception as e:
        missing.append(f"{key} ({e.__class__.__name__})")

if missing:
    raise RuntimeError(
        f"Thiếu hoặc chưa cấp quyền Colab secret(s): {missing}. "
        f"Panel trái > 🔑 Secrets, thêm đúng tên, BẬT công tắc 'Notebook access', rồi chạy lại cell này."
    )

# ghi ra .env để src/utils/env.py (và các training script chạy qua subprocess) tự động nạp
with open(".env", "w") as f:
    for k, v in loaded.items():
        f.write(f"{k}={v}\n")
        os.environ[k] = v

print("✓ Secrets loaded:", {k: f"***len={len(v)}" for k, v in loaded.items()})

In [ ]:
# --- Chọn config cho lần chạy này ---
# configs/config.yaml       = bản gốc (v1-feasibility, lambda_fine=1.0)
# configs/config_run_a.yaml = Run A (E-first giữ nguyên, chỉ giảm lambda_fine 1.0->0.5 + per-example loss)
CONFIG_PATH = "configs/config_run_a.yaml"
print("CONFIG_PATH:", CONFIG_PATH)

In [ ]:
# --- IN TOÀN BỘ CONFIG TRƯỚC KHI CHẠY BẤT KỲ TRAINING NÀO ---
import yaml, json

with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg_text = f.read()
cfg = yaml.safe_load(cfg_text)

print("=" * 80)
print(f"FULL CONFIG — {CONFIG_PATH} (raw)")
print("=" * 80)
print(cfg_text)
print("=" * 80)
print("Parsed config (sanity check):")
print(json.dumps(cfg, indent=2, ensure_ascii=False))
print("=" * 80)
print(f"Version      : {cfg['project']['version']}")
print(f"lambda_fine  : {cfg['training']['hier'].get('lambda_fine')}")
print(f"W&B project  : {cfg['wandb']['project']}  (entity theo API key)")
print(f"HF flat repo : {cfg['hf_hub']['flat_repo_id']}")
print(f"HF hier repo : {cfg['hf_hub']['hier_repo_id']}")

## Data — kéo thật **ViANLI** (thay cho synthetic placeholder trong repo)
`uitnlp/ViANLI` trên HF Hub (train 8012 / dev 1000 / test 1000, không gated). `scripts/prepare_vianli.py` tải `vianli_{train,dev,test}.jsonl`, đổi tên field (`uid→id`) và map nhãn (`entailment→E, contradiction→C, neutral→N`) rồi ghi đè `data/raw/*.jsonl`. Sau bước này, `load_or_generate()` trong `src/data/dataset.py` sẽ thấy file đã có và **không** sinh synthetic nữa — các script train dùng đúng data ViANLI này.

In [ ]:
!python3 scripts/prepare_vianli.py --config {CONFIG_PATH}
assert _exit_code == 0, "prepare_vianli.py FAILED — xem traceback ở trên (thường do mất mạng hoặc HF Hub rate limit)."
print()
!wc -l data/raw/train.jsonl data/raw/dev.jsonl data/raw/test.jsonl
!head -n 2 data/raw/train.jsonl

In [ ]:
# --- (Tùy chọn) DEBUG = True để chạy thử trên 200 mẫu/split cho nhanh trước khi chạy full ---
DEBUG = False
debug_flag = "--debug" if DEBUG else ""
print("DEBUG mode:", DEBUG)

## Step 1 & 3 — Flat (tùy chọn) + Hierarchical (bắt buộc) trên ViANLI
Mỗi run tự động: train → log metrics/loss/confusion matrix lên **W&B** → predict dev/test → push checkpoint + tokenizer + predictions per-sample lên **HF Hub** (gắn tag `v1`/`v2`/... trên cùng repo) → log lại `hf_repo_id`/`hf_revision` vào W&B run summary.

**Run A không cần train lại Flat** (baseline Flat không đổi) — mặc định `RUN_FLAT = False`, cell Step 1 sẽ **tải lại** predictions Flat đã train trước đó từ HF Hub thay vì train lại, để `evaluate_offline.py` vẫn so sánh được. Đổi `RUN_FLAT = True` nếu bạn thật sự muốn train lại Flat từ đầu.

**Quan trọng:** nếu 1 cell train báo `AssertionError` — nghiĩa là training thật sự đã crash. Đừng chạy tiếp các cell sau — cuộn lên xem traceback Python ngay phía trên dòng `assert` để biết lỗi thật (thường là OOM GPU, lỗi tải model, hoặc lỗi data).

In [ ]:
# --- Step 1: Flat CafeBERT (Run 1) — tùy chọn cho Run A ---
RUN_FLAT = False  # True = train lại Flat từ đầu; False = tải predictions Flat đã có sẵn trên HF Hub (đủ để so sánh)

if RUN_FLAT:
    !python3 -m src.training.train_flat --config {CONFIG_PATH} {debug_flag}
    assert _exit_code == 0, "train_flat.py FAILED — cuộn lên xem traceback Python ở trên, KHÔNG chạy các cell sau cho đến khi fix."
else:
    import pathlib, shutil
    from huggingface_hub import hf_hub_download
    flat_repo = cfg["hf_hub"]["flat_repo_id"]
    out_dir = pathlib.Path("outputs/predictions/flat"); out_dir.mkdir(parents=True, exist_ok=True)
    for split in ["dev", "test"]:
        p = hf_hub_download(repo_id=flat_repo, filename=f"predictions/{split}_predictions.csv", repo_type="model")
        shutil.copy(p, out_dir / f"{split}_predictions.csv")
        print(f"✓ tải {split}_predictions.csv từ {flat_repo} -> {out_dir}")

In [ ]:
# --- Step 3: Hierarchical CafeBERT 2-head (Run 2) ---
!python3 -m src.training.train_hierarchical --config {CONFIG_PATH} {debug_flag}
assert _exit_code == 0, "train_hierarchical.py FAILED — cuộn lên xem traceback Python ở trên, KHÔNG chạy các cell sau cho đến khi fix."

## Steps 2, 4A, 4B, 5 — offline (diagnostic, Hard/Soft, so sánh cuối)
Chạy offline từ prediction CSV đã lưu local (không train lại).

In [ ]:
!python3 scripts/evaluate_offline.py --config {CONFIG_PATH}
assert _exit_code == 0, "evaluate_offline.py FAILED — cuộn lên xem traceback ở trên."

In [ ]:
# --- In bảng so sánh cuối cùng ---
report_path = pathlib.Path("outputs/comparison/comparison_report.md")
if not report_path.exists():
    raise FileNotFoundError(
        f"{report_path} chưa tồn tại — nghĩa là evaluate_offline.py không đủ cả 2 file predictions "
        f"(outputs/predictions/flat/*.csv và outputs/predictions/hier/*.csv) để so sánh. Quay lại 2 cell "
        f"train_flat / train_hierarchical ở trên, xem có in ra dòng '[predict dev] -> ...csv' / "
        f"'[predict test] -> ...csv' không — nếu không có, training đã crash trước khi tới bước predict."
    )
with open(report_path, encoding="utf-8") as f:
    print(f.read())

## Kết quả & nơi xem lại
- **W&B**: metrics/loss/confusion matrix của từng run — xem URL in ra trong log của mỗi lệnh train ở trên (dạng `https://wandb.ai/<entity>/hierarchical-nli-e-first/runs/<id>`).
- **Hugging Face Hub**: checkpoint + tokenizer/config + predictions per-sample (đã train trên ViANLI thật). Mỗi lần chạy train **push vào đúng 1 repo cố định, không tạo repo mới** — chỉ gắn thêm git tag `v1`, `v2`, `v3`, ... tăng dần trên cùng repo đó, nên bản cũ vẫn tải lại được dù `main` đã có bản mới hơn:
  - `https://huggingface.co/trinhtrantran122/hier-nli-e-first-flat-cafebert`
  - `https://huggingface.co/trinhtrantran122/hier-nli-e-first-hier-cafebert`
  - Tải đúng 1 phiên bản cụ thể: `AutoModel.from_pretrained(repo_id, revision="v2")` (xem tab **Files and versions → v2** trên trang repo để biết `v2` ứng với commit nào).
- `outputs/comparison/comparison_report.md` trong `/content/hierarchical-nli-e-first/` — tải về qua panel Files (biểu tượng thư mục ở sidebar trái) nếu cần, trước khi ngắt runtime.